### PySpark Setup and Data Loading

First, we'll set up a SparkSession and load the flight dataset into a PySpark DataFrame.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, avg, count, lit, max

In [2]:
# Initialize SparkSession
spark = SparkSession.builder.appName("FlightAnalysis").getOrCreate()

# Load the dataset
file_path = "/content/Flight Dataset - CSV(in).csv"
df = spark.read.csv(file_path, header=True, inferSchema=True)

# Display schema and first few rows to understand the data
print("DataFrame Schema:")
df.printSchema()
print("\nFirst 5 rows of the DataFrame:")
df.show(5)

DataFrame Schema:
root
 |-- FL_DATE: string (nullable = true)
 |-- DEP_DELAY: integer (nullable = true)
 |-- ARR_DELAY: integer (nullable = true)
 |-- AIR_TIME: integer (nullable = true)
 |-- DISTANCE: integer (nullable = true)
 |-- DEP_TIME: double (nullable = true)
 |-- ARR_TIME: double (nullable = true)


First 5 rows of the DataFrame:
+--------+---------+---------+--------+--------+---------+---------+
| FL_DATE|DEP_DELAY|ARR_DELAY|AIR_TIME|DISTANCE| DEP_TIME| ARR_TIME|
+--------+---------+---------+--------+--------+---------+---------+
|1/1/2006|        5|       19|     350|    2475| 9.083333|12.483334|
|1/2/2006|      167|      216|     343|    2475|11.783334|15.766666|
|1/3/2006|       -7|       -2|     344|    2475| 8.883333|12.133333|
|1/4/2006|       -5|      -13|     331|    2475| 8.916667|    11.95|
|1/5/2006|       -3|      -17|     321|    2475|     8.95|11.883333|
+--------+---------+---------+--------+--------+---------+---------+
only showing top 5 rows


### Task 1: Flights Arriving Earlier Than Expected

Create a function that gives back how many flights arrived earlier than expected.

In [4]:
def count_early_arrivals(dataframe):
    """
    Counts the number of flights that arrived earlier than expected.
    An early arrival is defined as having an 'ArrDelay' (Arrival Delay) less than 0.

    Args:
        dataframe (pyspark.sql.DataFrame): The input DataFrame containing flight data.

    Returns:
        int: The number of flights that arrived earlier than expected.
    """
    # Filter for flights where 'ArrDelay' is negative (early arrival)
    # .na.drop() is used to exclude rows where ArrDelay is null, as a null delay is not an early arrival
    early_arrivals_count = dataframe.filter(col("ARR_DELAY") < 0).count()
    return early_arrivals_count

# Demonstrate the function
early_flights = count_early_arrivals(df)
print(f"Number of flights that arrived earlier than expected: {early_flights}")

Number of flights that arrived earlier than expected: 534655


### Task 2: Typical Departure Time for Long Flights

Create a function that determines the typical departure time for flights over 2000 miles.

In [5]:
def typical_departure_time_long_flights(dataframe):
    """
    Determines the typical (average) departure time for flights over 2000 miles.
    The 'DepTime' column represents the departure time.

    Args:
        dataframe (pyspark.sql.DataFrame): The input DataFrame containing flight data.

    Returns:
        float: The average departure time for flights over 2000 miles.
               Returns None if no such flights exist.
    """
    # Filter for flights with a distance greater than 2000 miles
    long_flights = dataframe.filter(col("DISTANCE") > 2000)

    # Calculate the average departure time
    # .na.drop() is used to exclude rows where DepTime is null
    avg_dep_time = long_flights.select(avg("DEP_TIME")).collect()[0][0]

    return avg_dep_time

# Demonstrate the function
avg_dep_time = typical_departure_time_long_flights(df)
if avg_dep_time is not None:
    print(f"Typical (average) departure time for flights over 2000 miles: {avg_dep_time:.2f}")
else:
    print("No flights found over 2000 miles.")

Typical (average) departure time for flights over 2000 miles: 13.97


### Task 3: Proportion of Flights with Long Arrival Delays

Create a function that gives back the proportion of flights that have arrival delays longer than 60 minutes.

In [6]:
def proportion_long_arrival_delays(dataframe):
    """
    Calculates the proportion of flights that have arrival delays longer than 60 minutes.

    Args:
        dataframe (pyspark.sql.DataFrame): The input DataFrame containing flight data.

    Returns:
        float: The proportion of flights with arrival delays longer than 60 minutes.
               Returns 0.0 if no flights are present.
    """
    total_flights = dataframe.count()

    if total_flights == 0:
        return 0.0

    # Filter for flights with 'ArrDelay' greater than 60 minutes
    long_delay_flights = dataframe.filter(col("ARR_DELAY") > 60)
    long_delay_count = long_delay_flights.count()

    proportion = long_delay_count / total_flights
    return proportion

# Demonstrate the function
prop_long_delay = proportion_long_arrival_delays(df)
print(f"Proportion of flights with arrival delays longer than 60 minutes: {prop_long_delay:.4f}")

Proportion of flights with arrival delays longer than 60 minutes: 0.0531


### Task 4: Average Airtime for Early Morning Departures

Create a function that gives the average airtime for flights that left earlier than 9:00 am (0900 in 24-hour format).

In [7]:
def average_airtime_early_departures(dataframe):
    """
    Calculates the average airtime for flights that departed earlier than 9:00 am.

    Args:
        dataframe (pyspark.sql.DataFrame): The input DataFrame containing flight data.

    Returns:
        float: The average airtime for flights departing before 9:00 am.
               Returns None if no such flights exist.
    """
    # Filter for flights where 'DepTime' is less than 900 (09:00 AM)
    early_morning_flights = dataframe.filter(col("DEP_TIME") < 900)

    # Calculate the average airtime for these flights
    # .na.drop() is used to exclude rows where AirTime is null
    avg_airtime = early_morning_flights.select(avg("AIR_TIME")).collect()[0][0]

    return avg_airtime

# Demonstrate the function
avg_airtime = average_airtime_early_departures(df)
if avg_airtime is not None:
    print(f"Average airtime for flights that left earlier than 9:00 AM: {avg_airtime:.2f} minutes")
else:
    print("No flights found departing before 9:00 AM.")

Average airtime for flights that left earlier than 9:00 AM: 105.81 minutes


### Task 5: Maximum Arrival Delay for On-Time Departures

Create a function that determines the maximum arrival delay for flights that did not experience a delay upon departure.

In [8]:
def max_arrival_delay_on_time_departure(dataframe):
    """
    Determines the maximum arrival delay for flights that did not experience a delay upon departure.
    A flight without a departure delay is defined as having 'DepDelay' (Departure Delay) less than or equal to 0.

    Args:
        dataframe (pyspark.sql.DataFrame): The input DataFrame containing flight data.

    Returns:
        float: The maximum arrival delay for flights with no departure delay.
               Returns None if no such flights exist.
    """
    # Filter for flights where 'DepDelay' is less than or equal to 0 (on-time or early departure)
    on_time_departures = dataframe.filter(col("DEP_DELAY") <= 0)

    # Find the maximum 'ArrDelay' among these flights
    # .na.drop() is used to exclude rows where ArrDelay is null
    max_arr_delay = on_time_departures.select(max("ARR_DELAY")).collect()[0][0]

    return max_arr_delay

# Demonstrate the function
max_delay = max_arrival_delay_on_time_departure(df)
if max_delay is not None:
    print(f"Maximum arrival delay for flights that departed on time (or early): {max_delay:.2f} minutes")
else:
    print("No flights found that departed on time (or early).")

Maximum arrival delay for flights that departed on time (or early): 701.00 minutes


### Stop Spark Session

It's good practice to stop the SparkSession when you're done with it.

In [9]:
# Stop the SparkSession
spark.stop()